# ⚡ Parallelization with RunnableParallel

The **parallelization** workflow runs several independent branches at the same
time, then merges their results. The branch set is **fixed when you write the
code** — you know which branches exist, and how many, before you see the input.

This notebook rebuilds
`05_AI_Agent_Fundamentals/4. Workflow_Pattern/3. Parallelization/` — the same
four parts, ending with the same Wikipedia-plus-web-search question about Nvidia
— using `RunnableParallel` instead of graph fan-out edges.

```
              ┌──▶ wikipedia search ──┐
question ──▶  ┤                       ├──▶ synthesize ──▶ answer
              └──▶ web search ────────┘
                (both run at once)
```

## Learning Objectives
In this notebook, you will learn:
1. **RunnableParallel** - run a fixed set of branches concurrently and collect a dict
2. **Why no reducers** - how LCEL sidesteps the `Annotated[list, operator.add]` problem
3. **Asymmetric branches** - fan out paths of different lengths and still join cleanly
4. **Concurrency control** - cap parallelism with `max_concurrency` in the run config
5. **Multi-source retrieval** - the pattern's most common real use, at the latency of the slowest source

## Prerequisites
- An `OPENAI_API_KEY` and a `TAVILY_API_KEY` in a `.env` file at the repo root
- `langchain >= 1.0`, plus `langchain-community` and `langchain-tavily` for Part 4
- Completion of `6.2_Routing.ipynb`

---

## 🔧 1. Setting Up the Environment

Parts 1 to 3 need nothing beyond `langchain-core`. Part 4 additionally needs the
Wikipedia and Tavily integrations, which live outside the core install.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and initialise the chat model
# ============================================================================
import time
import warnings

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel

warnings.filterwarnings("ignore")
load_dotenv()

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

print(f"✅ Model ready: {llm.__class__.__name__}")

In [ ]:
# ============================================================================
# OPTIONAL: Install the Part 4 integration packages
# ============================================================================
# Wikipedia and Tavily live outside the core install. Uncomment if needed.
# !pip install langchain-community wikipedia langchain-tavily

---

## 🐌 2. Part 1: The Sequential Baseline

First, the shape we are trying to beat. Four steps piped one after another, each
taking half a second of simulated work. Total time is the **sum** of the steps.

The LangGraph version used a `ReturnNodeValue` class appending to a reduced list.
In LCEL a step is just a function, and piping them makes the order explicit.

In [ ]:
# ============================================================================
# SEQUENTIAL CHAIN: Four steps, each waiting on the one before it
# ============================================================================
STEP_SECONDS = 0.5


def make_step(name: str):
    """Build a step that simulates half a second of work and logs its arrival."""

    def step(state: list) -> list:
        time.sleep(STEP_SECONDS)
        print(f"  Adding {name} to {state}")
        return [*state, name]

    return RunnableLambda(step)


sequential_chain = (
    make_step("step A") | make_step("step B") | make_step("step C") | make_step("step D")
)

start = time.perf_counter()
result = sequential_chain.invoke([])
elapsed = time.perf_counter() - start

print(f"\n📄 Execution order: {result}")
print(f"⏱️  Wall time: {elapsed:.2f}s (4 steps x {STEP_SECONDS}s = {4 * STEP_SECONDS:.1f}s)")

---

## ⚡ 3. Part 2: Fan-Out and Fan-In

Now B and C become independent branches. `RunnableParallel` runs every branch it
is given **concurrently** on a thread pool, and returns a **dict keyed by branch
name**.

That dict is the whole reason LCEL needs no reducers. In LangGraph, two nodes
writing to the same state key would overwrite each other unless you declared
`Annotated[list, operator.add]`. Here each branch owns its own key, so there is
nothing to merge and nothing to lose.

> **Key insight**: `RunnableParallel` is the *opposite* of `RunnableBranch`.
> The branch runs one of N. The parallel runs all of N.

In [ ]:
# ============================================================================
# PARALLEL CHAIN: A, then B and C at the same time, then D
# ============================================================================


def step_d(branches: dict) -> list:
    """Fan-in point: receives the dict of every branch result."""
    time.sleep(STEP_SECONDS)
    merged = branches["from_b"] + [item for item in branches["from_c"] if item not in branches["from_b"]]
    print(f"  Adding step D to {merged}")
    return [*merged, "step D"]


parallel_chain = (
    make_step("step A")
    | RunnableParallel(from_b=make_step("step B"), from_c=make_step("step C"))
    | RunnableLambda(step_d)
)

start = time.perf_counter()
result = parallel_chain.invoke([])
elapsed = time.perf_counter() - start

print(f"\n📄 Final state: {result}")
print(f"⏱️  Wall time: {elapsed:.2f}s — B and C overlapped, so this is ~3 steps, not 4")

---

## 🪜 4. Part 3: Asymmetric Parallel Paths

What if one branch is longer than the other? Path 1 is `B → X`, path 2 is just
`C`, and both must finish before D runs.

In LangGraph this needed the explicit list-syntax sync point
`add_edge(["x", "c"], "d")`. In LCEL it needs nothing special: a branch value can
itself be a multi-step chain, and `RunnableParallel` waits for **all** branches
regardless of their length.

In [ ]:
# ============================================================================
# ASYMMETRIC PATHS: Path 1 is B then X, path 2 is C alone, both join at D
# ============================================================================


def join_paths(branches: dict) -> list:
    """Wait for both paths, then merge and append D."""
    time.sleep(STEP_SECONDS)
    long_path, short_path = branches["path_1"], branches["path_2"]
    merged = long_path + [item for item in short_path if item not in long_path]
    print(f"  Adding step D to {merged}")
    return [*merged, "step D"]


asymmetric_chain = (
    make_step("step A")
    | RunnableParallel(
        path_1=make_step("step B") | make_step("step X"),  # two steps deep
        path_2=make_step("step C"),                        # one step deep
    )
    | RunnableLambda(join_paths)
)

start = time.perf_counter()
result = asymmetric_chain.invoke([])
elapsed = time.perf_counter() - start

print(f"\n📄 Final state: {result}")
print(f"⏱️  Wall time: {elapsed:.2f}s — paced by the LONGER path, not the sum")

---

## 🎛️ 5. Controlling Concurrency

By default `RunnableParallel` gives every branch its own thread. When branches
call a rate-limited API you will want a ceiling, which is set per invocation
through the run config rather than at build time.

In [ ]:
# ============================================================================
# MAX CONCURRENCY: Cap how many branches run at once
# ============================================================================
six_branches = RunnableParallel(
    **{f"branch_{i}": make_step(f"work {i}") for i in range(6)}
)

start = time.perf_counter()
six_branches.invoke([])
unbounded = time.perf_counter() - start

start = time.perf_counter()
six_branches.invoke([], config={"max_concurrency": 2})
throttled = time.perf_counter() - start

print(f"\n⏱️  6 branches unbounded:        {unbounded:.2f}s")
print(f"⏱️  6 branches, max 2 at a time: {throttled:.2f}s")

---

## 🌐 6. Part 4: Multi-Source Retrieval

The real payoff. Wikipedia gives foundational background, web search gives
current news, and neither depends on the other — so running them together costs
the latency of the **slower** source rather than the sum of both.

Note the modern import: `TavilySearch` from the `langchain-tavily` package. The
`TavilySearchResults` class used in the original LangGraph notebook came from
`langchain_community` and is deprecated.

In [ ]:
# ============================================================================
# SEARCH BRANCHES: Two independent retrieval functions
# ============================================================================
from langchain_community.document_loaders import WikipediaLoader
from langchain_tavily import TavilySearch


def search_wikipedia(payload: dict) -> str:
    """Load up to 2 Wikipedia articles matching the question."""
    print("  --Getting Info from Wikipedia--")
    docs = WikipediaLoader(query=payload["question"], load_max_docs=2).load()
    return "\n\n---\n\n".join(
        f"Content: {d.page_content}\nSource: {d.metadata['source']}" for d in docs
    )


def search_web(payload: dict) -> str:
    """Search the web via Tavily for recent information."""
    print("  --Getting Info from the Web--")
    response = TavilySearch(max_results=3).invoke({"query": payload["question"]})
    return "\n\n---\n\n".join(
        f"Content: {r['content']}\nSource: {r['url']}" for r in response["results"]
    )


print("✅ Search branches defined")

The synthesis step receives the parallel dict and folds both sources into one
prompt. Because each source has its own key, the prompt can label them.

In [ ]:
# ============================================================================
# SYNTHESIS CHAIN: Fan-in — one LLM call over both sources
# ============================================================================
synthesis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using these context documents.\n\n"
            "Wikipedia context:\n{wikipedia}\n\nWeb search context:\n{web}",
        ),
        ("human", "{question}"),
    ]
)

parallel_search_chain = (
    RunnableParallel(
        wikipedia=RunnableLambda(search_wikipedia),
        web=RunnableLambda(search_web),
        question=lambda payload: payload["question"],  # carry the question forward
    )
    | synthesis_prompt
    | llm
    | StrOutputParser()
)

print("✅ Parallel search chain assembled")

---

## 🧪 7. Running the Multi-Source Chain

The same question as the LangGraph notebook.

In [ ]:
# ============================================================================
# TEST: Ask a question that benefits from both a reference and recent news
# ============================================================================
question = "Tell me about Nvidia and its growth"

print(f"Question: {question}")
print("=" * 60)

start = time.perf_counter()
answer = parallel_search_chain.invoke({"question": question})
elapsed = time.perf_counter() - start

print(f"\n⏱️  Wall time: {elapsed:.2f}s")
print("=" * 60)
print("AGENT'S ANSWER")
print("=" * 60)
print(answer)

---

## 📝 Summary

We rebuilt the LangGraph parallelization workflow in LCEL with `RunnableParallel`.

### 1. The translation table

| LangGraph | LCEL |
|---|---|
| two `add_edge(START, node)` calls for fan-out | `RunnableParallel(a=..., b=...)` |
| two `add_edge(node, join)` calls for fan-in | the dict `RunnableParallel` returns |
| `Annotated[list, operator.add]` reducer | **not needed** — each branch owns its own key |
| `add_edge(["x", "c"], "d")` sync point | a branch whose value is itself a multi-step chain |
| no built-in throttle | `config={"max_concurrency": N}` |

### 2. What to remember
- **Reducers disappear.** The reducer existed to stop concurrent writes from
  clobbering one another. A keyed dict has no collision to resolve.
- **Latency is the slowest branch.** That holds whether branches are equal
  length or not.
- **The branch set is fixed at build time.** You write the branch names into the
  code. If the number of branches depends on the input, you have crossed into
  orchestrator-worker territory.

### 3. Parallelization vs orchestrator-worker
Ask the question the LangGraph README poses: *could I have hardcoded the exact
number of parallel branches before seeing this input?* Yes means
`RunnableParallel`. No means the next notebook.

### Next Steps
- `6.4_Orchestrator_Worker.ipynb` — fan out to a count the input decides at run time
- Compare against the LangGraph original in
  `05_AI_Agent_Fundamentals/4. Workflow_Pattern/3. Parallelization/`